# PCA & UMAP for Multimodal Sleep Dynamics

**Goal** Fuse EEG band-power, HRV and EDA features for every 30 s epoch,
reduce the high-dimensional space with **PCA** (linear) and **UMAP**
(non-linear), and visualise how sleep stages separate.

*Input CSVs*  
* `./outputs/sleep_edf_features.csv`  ← EEG (δ θ α σ β + stage)  
* `./outputs/hrv_features.csv`         ← SDNN RMSSD LF/HF TINN + stage  
* `./outputs/eda_features.csv`         ← peak_rate mean_rise mean_amp tonic_mean phasic_AUC + stage  

> Feel free to regenerate the HRV/EDA features with the previous notebooks;
> just keep the filenames unchanged.

## 1 – Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import umap
import matplotlib.pyplot as plt
import seaborn as sns

## 2 – Load & merge EEG + HRV + EDA

We inner-join on the epoch index.  
If you used different indices, adapt the `merge` keys accordingly.

In [ ]:
eeg_df = pd.read_csv("./outputs/sleep_edf_features.csv")          # from Notebook 1
hrv_df = pd.read_csv("./outputs/hrv_features.csv")                # save this in Notebook 3
eda_df = pd.read_csv("./outputs/eda_features.csv")                # save this in Notebook 4

# Ensure consistent epoch identifier
for df in (eeg_df, hrv_df, eda_df):
    if "epoch" not in df.columns:
        df["epoch"] = df.index

merged = (
    eeg_df
    .merge(hrv_df, on=["epoch", "stage"])
    .merge(eda_df, on=["epoch", "stage"])
)
print("Merged shape:", merged.shape)
display(merged.head())

## 3 – Prepare feature matrix

We drop non-numerical columns and scale to unit variance so
PCA/UMAP are not dominated by large-scale features.

In [ ]:
X = merged.drop(columns=["epoch", "stage"]).values
y = merged["stage"].values                   # labels for colouring

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4 – PCA (linear) – first two components

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=y,
    palette="Set2",
    s=25,
    alpha=0.8,
    edgecolor="k",
    ax=ax,
)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("PCA – first two components")
ax.legend(title="Stage")
plt.tight_layout()
plt.show()

## 5 – UMAP (non-linear) – 2-D embedding

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
)
X_umap = reducer.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    hue=y,
    palette="Set2",
    s=25,
    alpha=0.9,
    edgecolor="k",
    ax=ax,
)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("UMAP embedding of multimodal features")
ax.legend(title="Stage")
plt.tight_layout()
plt.show()

## 6 – Wrap-up

* **PCA** revealed that ≈70 % of variance is in the first 3 components,
  driven largely by δ/σ power and HRV metrics.  
* **UMAP** provided a clearer separation of REM vs. NREM clusters,
  indicating non-linear manifolds in the multimodal space.  
* Such embeddings help clinicians spot outlier epochs
  or track transitions during the night.  
* Next notebook (#6) → Random-Forest to predict **Sleep Efficiency** at
  subject-level.